# Explore GT export

Quick sanity checks on a faceiq-labs export. Run the export first (see README), then execute cells top to bottom.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

from faceiq_pref.data import load_export

EXPORT_DIR = ROOT / 'data' / 'exports' / 'cmr1mr0m7000196d57zi3vcgn'
export = load_export(EXPORT_DIR)
export.manifest['counts']

In [ ]:
import pandas as pd

matchups = export.all_matchups()
df = pd.DataFrame([m.__dict__ for m in matchups])
print(len(df), 'matchups')
df['final_outcome'].value_counts()

In [ ]:
# Confidence distribution and human-audit overlap
print(df['confidence'].value_counts(dropna=False))
print()
print('human-labeled rows:', df['human_labeled_at'].notna().sum())
print('human overrides:', df['is_human_override'].sum())

In [ ]:
# Image coverage + a peek at a few faces
from PIL import Image
import matplotlib.pyplot as plt

faces = export.faces()
present, missing = export.image_coverage(faces)
print(f'{present}/{len(faces)} images present, {len(missing)} missing')

sample = [f for f in list(faces.values())[:6] if export.image_path(f).exists()]
fig, axes = plt.subplots(1, len(sample), figsize=(3 * len(sample), 3))
for ax, face in zip(axes, sample):
    ax.imshow(Image.open(export.image_path(face)))
    ax.set_title(f'{face.gender} D{face.decile_bin}', fontsize=9)
    ax.axis('off')

## Cross-check: choix (ILSR) vs our MM refit

Independent validation of `src/faceiq_pref/bt.py`: fit Bradley-Terry with `choix.ilsr_pairwise` on the same win/loss edges and compare rankings against the shipped `artifacts/bt-refit-v1/ratings.csv`.

Notes from the 2026-07-04 run:

- `alpha` semantics differ between libraries — choix's `alpha=0.01` shrinks much harder than our virtual-game `alpha=0.01`. With `alpha=1e-4` the two rankings agree at Spearman rho ~0.9995 per gender, converging toward the same unregularized MLE.
- Ties (113 rows) are dropped here (choix cannot weight edges); re-fitting our MM on the same tie-dropped edges reproduces the shipped refit at rho = 0.99997, so tie handling is immaterial.

In [ ]:
import choix
from scipy.stats import spearmanr

ratings = pd.read_csv(ROOT / 'artifacts' / 'bt-refit-v1' / 'ratings.csv')

for gender in ['female', 'male']:
    ours = ratings[ratings['gender'] == gender].set_index('faceId')['theta']
    face_ids = sorted(ours.index)  # faces actually scored (post under-connected drop)
    index = {fid: i for i, fid in enumerate(face_ids)}
    data = []
    for m in matchups:
        if m.gender != gender or m.is_tie:
            continue
        if m.face_a_id not in index or m.face_b_id not in index:
            continue
        winner = index[m.final_winner_face_id]
        loser = index[m.face_b_id if m.final_winner_face_id == m.face_a_id else m.face_a_id]
        data.append((winner, loser))
    # alpha=1e-4: choix's regularization is a different (stronger) prior than our
    # virtual-game alpha; near-zero values compare the underlying MLEs fairly.
    theta_choix = choix.ilsr_pairwise(len(face_ids), data, alpha=1e-4)
    rho, _ = spearmanr(theta_choix, [ours[fid] for fid in face_ids])
    print(f'{gender}: spearman(choix, ours) = {rho:.6f}  ({len(face_ids)} faces, {len(data)} pairs)')